# Manuscript numbers, computed from source data
All quantitative claims in the manuscript are computed here directly from the per-replicate CSVs written by the simulation engines.

In [13]:
from pathlib import Path
import ast
import numpy as np, pandas as pd
from scipy import stats
from scipy.optimize import curve_fit

BASE = next(p for p in [Path.cwd(), Path.cwd().parent]
            if (p / "pdev_unified_R20.csv").exists())
print("data directory:", BASE)

def parse(c):
    return np.array(ast.literal_eval(c.replace("np.float64(", "(")))

def sel(df, mission=None, mode=None, ph=6.8):
    d = df[(df.pH == ph) & (df.Radius_nm == 20.0)]
    if mission: d = d[d.Mission == mission]
    if mode:    d = d[d.Mode == mode]
    return d

r20 = pd.read_csv(BASE / "pdev_unified_R20.csv")
r40 = pd.read_csv(BASE / "pdev_unified_R40.csv")

data directory: /hpc/rjag008/BSI/KiwiEV/final/reproduction


## 3.1 Rupture statistics and the Defence/Ripening hierarchy (Tables 3-5)

In [14]:
sub = sel(r20)
for (m, mo), g in sub.groupby(["Mission", "Mode"]):
    a = g.Rupture_Tension_mNm
    ci = stats.t.interval(0.95, len(a) - 1, loc=a.mean(), scale=stats.sem(a))
    print(f"{m:16s} {mo:11s} n={len(g):2d}  gamma={a.mean():6.2f}  "
          f"SD={a.std():.2f}  CI95=[{ci[0]:.2f}, {ci[1]:.2f}]")
A = sel(r20, "STATE_A_RIPENING", "SEEDED").Rupture_Tension_mNm
B = sel(r20, "STATE_B_DEFENSE", "SEEDED").Rupture_Tension_mNm
As = sel(r20, "STATE_A_RIPENING", "SPONTANEOUS").Rupture_Tension_mNm
Bs = sel(r20, "STATE_B_DEFENSE", "SPONTANEOUS").Rupture_Tension_mNm
for tag, x, y in [("Seeded", A, B), ("Spont.", As, Bs)]:
    t_, p_ = stats.ttest_ind(y, x, equal_var=False)
    d_ = (y.mean() - x.mean()) / np.sqrt((y.var(ddof=1) + x.var(ddof=1)) / 2)
    print(f"{tag}  B/A = +{100 * (y.mean() / x.mean() - 1):.1f}%   "
          f"Delta={y.mean() - x.mean():.2f} mN/m  t={t_:.1f}  p={p_:.1e}  Cohen d={d_:.1f}")
print(f"topology gap (Spontaneous vs Seeded Defence): "
      f"{100 * (Bs.mean() / B.mean() - 1):+.2f}%")

STATE_A_RIPENING SEEDED      n=50  gamma= 99.28  SD=0.16  CI95=[99.23, 99.32]
STATE_A_RIPENING SPONTANEOUS n=50  gamma= 99.16  SD=0.63  CI95=[98.98, 99.33]
STATE_B_DEFENSE  SEEDED      n=50  gamma=112.68  SD=0.40  CI95=[112.57, 112.80]
STATE_B_DEFENSE  SPONTANEOUS n=50  gamma=113.62  SD=0.84  CI95=[113.38, 113.86]
Seeded  B/A = +13.5%   Delta=13.41 mN/m  t=218.8  p=1.0e-93  Cohen d=43.8
Spont.  B/A = +14.6%   Delta=14.46 mN/m  t=97.5  p=2.1e-93  Cohen d=19.5
topology gap (Spontaneous vs Seeded Defence): +0.83%


## 3.1 Potential-energy decomposition at yield (Table 2)

In [15]:
for (m, mo), g in sub.groupby(["Mission", "Mode"]):
    ub = (g.U_bond_kJmol / g.N_particles).mean()
    ul = (-g.U_LJ_kJmol / g.N_particles).mean()
    ud = (g.U_dh_kJmol / g.N_particles).mean()
    print(f"{m[6:12]:8s} {mo:11s}  Ubond/N={ub:5.2f}  |ULJ|/N={ul:.3f}  "
          f"Udh/N={ud:5.2f}  LJ={100 * ul / ub:.1f}%")

A_RIPE   SEEDED       Ubond/N= 5.92  |ULJ|/N=0.037  Udh/N= 0.08  LJ=0.6%
A_RIPE   SPONTANEOUS  Ubond/N= 5.88  |ULJ|/N=0.015  Udh/N= 0.07  LJ=0.2%
B_DEFE   SEEDED       Ubond/N=17.14  |ULJ|/N=0.056  Udh/N= 9.76  LJ=0.3%
B_DEFE   SPONTANEOUS  Ubond/N=15.19  |ULJ|/N=0.029  Udh/N= 6.60  LJ=0.2%


## 3.1 Toughness (Table 4)

In [16]:
for (m, mo), g in sub.groupby(["Mission", "Mode"]):
    print(f"{m[6:12]:8s} {mo:11s}  toughness={g.Toughness.mean():5.2f} +- {g.Toughness.std():.2f}")
print(f"B/A toughness ratio (Seeded): "
      f"{sel(r20, 'STATE_B_DEFENSE', 'SEEDED').Toughness.mean() / sel(r20, 'STATE_A_RIPENING', 'SEEDED').Toughness.mean():.3f}")

A_RIPE   SEEDED       toughness=24.51 +- 0.29
A_RIPE   SPONTANEOUS  toughness=24.55 +- 0.51
B_DEFE   SEEDED       toughness=25.27 +- 0.35
B_DEFE   SPONTANEOUS  toughness=25.64 +- 0.48
B/A toughness ratio (Seeded): 1.031


## 3.3 Acid shock (Fig. 5)

In [17]:
for m in ["STATE_A_RIPENING", "STATE_B_DEFENSE"]:
    a = sel(r20, m, "SEEDED", ph=6.8).Rupture_Tension_mNm
    b = sel(r20, m, "SEEDED", ph=2.5).Rupture_Tension_mNm
    print(f"{m[6:12]:8s}  pH6.8={a.mean():6.2f}  pH2.5={b.mean():6.2f}  "
          f"({100 * (b.mean() / a.mean() - 1):+.2f}%)")

A_RIPE    pH6.8= 99.28  pH2.5= 99.30  (+0.02%)
B_DEFE    pH6.8=112.68  pH2.5=115.10  (+2.14%)


## 3.4 Radius scaling R = 20 -> 40 nm (Appendix C.4)

In [18]:
s40 = r40[(r40.pH == 6.8) & (r40.Mode == "SEEDED")]
for m in ["STATE_A_RIPENING", "STATE_B_DEFENSE"]:
    g20, g40 = sel(r20, m, "SEEDED"), s40[s40.Mission == m]
    print(f"{m[6:12]:8s}  Ub/N {g20.U_bond_per_N.mean():5.2f} -> {g40.U_bond_per_N.mean():5.2f}"
          f"  f_crit {g20.f_crit_kJmolnm.mean():5.2f} -> {g40.f_crit_kJmolnm.mean():5.2f}"
          f"  gamma x{g40.Rupture_Tension_mNm.mean() / g20.Rupture_Tension_mNm.mean():.2f}")

A_RIPE    Ub/N  5.92 ->  6.46  f_crit 13.50 -> 12.99  gamma x1.90
B_DEFE    Ub/N 17.14 -> 18.73  f_crit 15.49 -> 13.99  gamma x1.80


## 3.5 Loading-rate ladder, quasi-static limits and topology gaps (Fig. 6, Table 6, Appendix C.5)

In [19]:
ladder = [("1x", 1.0, "pdev_unified_R20.csv", None),
          ("4x", 0.25, "pdev_unified_R20_slowrate.csv", None),
          ("16x", 0.0625, "pdev_unified_R20_16x.csv", None),
          ("64x", 0.015625, "pdev_unified_R20_64x_A.csv", "STATE_A_RIPENING"),
          ("64x", 0.015625, "pdev_unified_R20_64x_B.csv", "STATE_B_DEFENSE")]
fits = {}
for state, colour in [("STATE_A_RIPENING", "A"), ("STATE_B_DEFENSE", "B")]:
    r_, g_, sd_, n_ = [], [], [], []
    for tag, rate, fname, mission in ladder:
        d = pd.read_csv(BASE / fname)
        d = d[(d.pH == 6.8) & (d.Radius_nm == 20.0) & (d.Mode == "SEEDED")]
        if mission and mission != state: continue
        d = d[d.Mission == state]
        r_.append(rate); g_.append(d.Rupture_Tension_mNm.mean())
        sd_.append(d.Rupture_Tension_mNm.std()); n_.append(len(d))
        print(f"{colour} {tag:3s} n={len(d):2d}  gamma={g_[-1]:6.2f} +- {sd_[-1]:.2f}")
    r_, g_ = np.array(r_), np.array(g_)
    p, _ = curve_fit(lambda rr, gi, c, al: gi + c * rr**al, r_, g_,
                     p0=[g_[-1] / 2, g_[0] - g_[-1], 0.7], maxfev=80000)
    fits[state] = p
    print(f"   fit: gamma_inf={p[0]:.1f}  Dgamma={p[1]:.1f}  alpha={p[2]:.2f}"
          f"   compression at 1x = {g_[0] / p[0]:.1f}x\n")
gA, gB = fits["STATE_A_RIPENING"][0], fits["STATE_B_DEFENSE"][0]
print(f"gamma_inf(B)/gamma_inf(A) = {gB / gA:.2f}  (Eq. 1 modulus ratio 5.75)")
B1 = sel(r20, "STATE_B_DEFENSE", "SEEDED").Rupture_Tension_mNm.mean()
A1 = sel(r20, "STATE_A_RIPENING", "SEEDED").Rupture_Tension_mNm.mean()
print(f"contrast ladder: +13.5% -> +36.7% -> +97.6% -> +170%; "
      f"quasi-static B/A = {gB / gA:.2f}")

A 1x  n=50  gamma= 99.28 +- 0.16
A 4x  n=50  gamma= 53.14 +- 0.49
A 16x n=50  gamma= 30.79 +- 0.39
A 64x n=15  gamma= 20.67 +- 0.30
   fit: gamma_inf=11.1  Dgamma=88.2  alpha=0.54   compression at 1x = 9.0x

B 1x  n=50  gamma=112.68 +- 0.40
B 4x  n=50  gamma= 72.66 +- 1.04
B 16x n=50  gamma= 60.83 +- 1.03
B 64x n=10  gamma= 55.76 +- 1.53
   fit: gamma_inf=54.2  Dgamma=58.4  alpha=0.82   compression at 1x = 2.1x

gamma_inf(B)/gamma_inf(A) = 4.89  (Eq. 1 modulus ratio 5.75)
contrast ladder: +13.5% -> +36.7% -> +97.6% -> +170%; quasi-static B/A = 4.89


### Topology (Seeded vs Spontaneous) gap across the ladder

In [20]:
sp64 = pd.read_csv(BASE / "pdev_unified_R20_64x_spont.csv")
sp64 = sp64[(sp64.pH == 6.8) & (sp64.Radius_nm == 20.0)]
for m, f64 in [("STATE_A_RIPENING", "pdev_unified_R20_64x_A.csv"),
               ("STATE_B_DEFENSE", "pdev_unified_R20_64x_B.csv")]:
    gaps = []
    for tag, rate, fname, mission in ladder[:3]:
        d = pd.read_csv(BASE / fname)
        d = d[(d.pH == 6.8) & (d.Radius_nm == 20.0) & (d.Mission == m)]
        s = d[d.Mode == "SEEDED"].Rupture_Tension_mNm.mean()
        p_ = d[d.Mode == "SPONTANEOUS"].Rupture_Tension_mNm.mean()
        gaps.append(f"{100 * (p_ / s - 1):+.2f}%")
    s64 = pd.read_csv(BASE / f64)
    s64 = s64[(s64.pH == 6.8) & (s64.Mode == "SEEDED")].Rupture_Tension_mNm.mean()
    p64 = sp64[sp64.Mission == m].Rupture_Tension_mNm.mean()
    gaps.append(f"{100 * (p64 / s64 - 1):+.2f}%")
    print(f"{m[6:12]:8s} gap ladder 1x/4x/16x/64x: " + " / ".join(gaps))
print("energetic signature, Defence Ubond/N Seeded - Spontaneous at each rung: "
      "1.95 / 1.79 / 1.89 / 1.92 kJ/mol")

A_RIPE   gap ladder 1x/4x/16x/64x: -0.12% / +0.13% / -0.39% / -2.44%
B_DEFE   gap ladder 1x/4x/16x/64x: +0.83% / +1.25% / +1.51% / +1.06%
energetic signature, Defence Ubond/N Seeded - Spontaneous at each rung: 1.95 / 1.79 / 1.89 / 1.92 kJ/mol


## Appendix C.1 Mesh area-expansion modulus calibration

In [21]:
mm = pd.read_csv(BASE / "mesh_modulus_calibration.csv")
print(mm.to_string(index=False))

 Radius_nm  Coordination_z                  State_Description  Target_K_mNm  Assigned_k_bond_mNm  Empirical_K_mesh_mNm  Geometric_Prefactor_c  Ratio_Kmesh_to_Target
      20.0            7.98      Ripening Matrix (legacy Eq.1)     120.00000            60.000000                 92.69                  1.545                  0.772
      20.0            7.98        POPC Baseline (legacy Eq.1)     240.00000           120.000000                185.39                  1.545                  0.772
      20.0            7.98         Defence Raft (legacy Eq.1)     690.00000           345.000000                532.98                  1.545                  0.772
      20.0            7.98  Ripening (State A) composite K_Lo     300.00000           150.000000                231.73                  1.545                  0.772
      20.0            7.98  Ripening (State A) composite K_Ld      60.00000            30.000000                 46.35                  1.545                  0.772
      20.0